<a href="https://colab.research.google.com/github/StephenOla/Tuberculosis-Detection/blob/Main/Output_Level_Ensemble_Tuberculosis_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IMPORTING LIBRARIES**

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import cv2
from matplotlib import gridspec
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory, image
from tensorflow.keras.applications import ResNet50, MobileNetV2, VGG16
from tensorflow.keras.models import Sequential, Model, load_model, clone_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Concatenate, Conv2D, MaxPooling2D
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomContrast
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, f1_score, ConfusionMatrixDisplay


from google.colab import drive

**MOUNTING DRIVE AND READING DATASET**

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_path = "/content/drive/MyDrive/Datasets/Chest_Xrays/"
for folder in os.listdir(data_path):
  folder_total_content = len(os.listdir(data_path+"/"+folder))
  print(f"{folder} has : {folder_total_content} files")
  print()

Normal Chest X-rays has : 3965 files

TB Chest X-rays has : 3194 files



**DATASET PREPROCESSING**

In [ ]:
import os
import shutil
import random

def split_dataset(source_dir, output_dir, train_split=0.7, val_split=0.2, test_split=0.1, seed=42):
    if train_split + val_split + test_split != 1.0:
        print("Error: train_split, val_split, and test_split must sum to 1.0")
        test_split = 1.0 - train_split - val_split
        print(f"test_split has been adjusted to {test_split} to ensure the sum is 1.0")

    random.seed(seed)
    class_names = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]

    for class_name in class_names:
        class_path = os.path.join(source_dir, class_name)
        images = os.listdir(class_path)
        random.shuffle(images)

        n_total = len(images)
        n_train = int(train_split * n_total)
        n_val = int(val_split * n_total)
        n_test = n_total - n_train - n_val

        splits = {
            'train': images[:n_train],
            'val': images[n_train:n_train + n_val],
            'test': images[n_train + n_val:]
        }

        for split, split_images in splits.items():
            split_class_dir = os.path.join(output_dir, split, class_name)
            os.makedirs(split_class_dir, exist_ok=True)

            for img in split_images:
                src = os.path.join(class_path, img)
                dst = os.path.join(split_class_dir, img)
                shutil.copy2(src, dst)*

source_path = "/content/drive/MyDrive/Datasets/Chest_Xrays/"
output_path = "/content/drive/MyDrive/Datasets/Chest_Xrays_Split"

split_dataset(source_path, output_path)


SyntaxError: invalid syntax (ipython-input-97063641.py, line 37)

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)


base_dir = '/content/drive/MyDrive/Datasets/Chest_Xrays_Split'

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_dir, 'training'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_dir, 'validation'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_dir, 'testing'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
print(train_data.class_names)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1)
])

augmented_train_data = train_data.map(lambda x, y: (data_augmentation(x, training=True), y))

for images, _ in augmented_train_data.take(1):
    plt.figure(figsize=(10, 10))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.axis("off")

**RESNET50 MODEL BUILD**

In [ ]:
resnet_inputs = Input(shape=(224, 224, 3))
base_model = ResNet50(include_top=False, weights='imagenet', input_tensor=resnet_inputs)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dense(1, activation='sigmoid')(x)

resnet_model = Model(resnet_inputs, x)

resnet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


history_resnet = resnet_model.fit(
    augmented_train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
resnet_model.evaluate(test_data)

**RESNET50 MODEL EVALUATION**

In [ ]:
plt.plot(history_resnet.history['accuracy'], label='Train Accuracy')
plt.plot(history_resnet.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history_resnet.history['loss'], label='Train Loss')
plt.plot(history_resnet.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels in test_data:
    preds = resnet_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

class_names = ['Normal', 'Tuberculosis']
class_result = classification_report(y_true, y_pred, target_names=class_names)
print(class_result)

In [ ]:
class_names = ['Normal', 'Tuberculosis']
y_true = []
y_pred = []

for images, labels in test_data:
    preds = resnet_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
resnet_model.save(os.path.join('/content/drive/MyDrive/Models', 'resnet_tb_model.keras'))

In [ ]:
with open('/content/drive/MyDrive/Models/resnet_history.pkl', 'wb') as f:
    pickle.dump(history_resnet.history, f)


In [ ]:
resnet_model = load_model('/content/drive/MyDrive/Models/resnet_tb_model.keras')

In [ ]:
with open('/content/drive/MyDrive/Models/resnetV2_history.pkl', 'wb') as f:
    pickle.dump(history_resnetV2.history, f)

**MOBILENET MODEL BUILD**

In [ ]:
mobile_inputs = Input(shape=(224, 224, 3))
mobile_base_model = MobileNetV2(include_top=False, weights='imagenet', input_tensor=mobile_inputs)
mobile_base_model.trainable = False

x = mobile_base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dense(1, activation='sigmoid')(x)

mobile_model = Model(mobile_inputs, x)

mobile_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


history_mobile = mobile_model.fit(
    augmented_train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
mobile_model.save('/content/drive/MyDrive/Models/mobile_tb_model.keras')

**MOBILENET MODEL EVALUATION**

In [ ]:
mobile_model.evaluate(test_data)

In [ ]:
plt.plot(history_mobile.history['accuracy'], label='Train Accuracy')
plt.plot(history_mobile.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history_mobile.history['loss'], label='Train Loss')
plt.plot(history_mobile.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels in test_data:
    preds = mobile_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

class_names = ['Normal', 'Tuberculosis']
class_result = classification_report(y_true, y_pred, target_names=class_names)
print(class_result)

In [ ]:
class_names = ['Normal', 'Tuberculosis']
y_true = []
y_pred = []

for images, labels in test_data:
    preds = mobile_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
with open('/content/drive/MyDrive/Models/mobile_history.pkl', 'wb') as f:
    pickle.dump(history_mobile.history, f)

**VGG16 MODEL BUILD**

In [ ]:
vgg_inputs = Input(shape=(224, 224, 3))
vgg_base_model = VGG16(include_top=False, weights='imagenet', input_tensor=vgg_inputs)
vgg_base_model.trainable = False

x = vgg_base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dense(1, activation='sigmoid')(x)

vgg_model = Model(vgg_inputs, x)

vgg_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


history_vgg = vgg_model.fit(
    augmented_train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
vgg_model.save('/content/drive/MyDrive/Models/vgg_tb_model.keras')

In [ ]:
with open('/content/drive/MyDrive/Models/vgg_history.pkl', 'wb') as f:
    pickle.dump(history_vgg.history, f)

**VGG16 MODEL EVALUATION**

In [ ]:
vgg_model.evaluate(test_data)

In [ ]:
plt.plot(history_vgg.history['accuracy'], label='Train Accuracy')
plt.plot(history_vgg.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history_vgg.history['loss'], label='Train Loss')
plt.plot(history_vgg.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels in test_data:
    preds = vgg_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

class_names = ['Normal', 'Tuberculosis']
class_result = classification_report(y_true, y_pred, target_names=class_names)
print(class_result)

In [ ]:
class_names = ['Normal', 'Tuberculosis']
y_true = []
y_pred = []

for images, labels in test_data:
    preds = vgg_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

**ENSEMBLE MODEL BUILD**

In [ ]:
resnet_model = load_model('/content/drive/MyDrive/Models/resnet_tb_model.keras')
vgg_model = load_model('/content/drive/MyDrive/Models/vgg_tb_model.keras')
mobile_model = load_model('/content/drive/MyDrive/Models/mobile_tb_model.keras')

for model in [resnet_model, vgg_model, mobile_model]:
    model.trainable = False

ensemble_input = Input(shape=(224, 224, 3))

resnet_output = Model(resnet_model.input, resnet_model.output, name="resnet_sub")(ensemble_input)
vgg_output = Model(vgg_model.input, vgg_model.output, name="vgg_sub")(ensemble_input)
mobile_output = Model(mobile_model.input, mobile_model.output, name="mobile_sub")(ensemble_input)

merged = Concatenate()([
    resnet_output,
    vgg_output,
    mobile_output
])
x = Dropout(0.5)(merged)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

ensemble_model = Model(inputs=ensemble_input, outputs=output)

ensemble_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
history_ensemble = ensemble_model.fit(
    augmented_train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
)

In [ ]:
ensemble_model.save('/content/drive/MyDrive/Models/Ensemble_tb_model.keras')

In [ ]:
with open('/content/drive/MyDrive/Models/Ensemble_history.pkl', 'rb') as f:
    history_ensemble = pickle.load(f)

In [ ]:
with open('/content/drive/MyDrive/Models/Ensemble_history.pkl', 'wb') as f:
    pickle.dump(history_ensemble.history, f)

**ENSEMBLE MODEL EVALUATION**

In [ ]:
ensemble_model.evaluate(test_data)

In [ ]:
plt.plot(history_ensemble.history['accuracy'], label='Train Accuracy')
plt.plot(history_ensemble.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history_ensemble.history['loss'], label='Train Loss')
plt.plot(history_ensemble.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels in test_data:
    preds = ensemble_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

class_names = ['Normal', 'Tuberculosis']
class_result = classification_report(y_true, y_pred, target_names=class_names)
print(class_result)

In [ ]:
class_names = ['Normal', 'Tuberculosis']
y_true = []
y_pred = []

for images, labels in test_data:
    preds = ensemble_model.predict(images)
    preds = (preds > 0.5).astype(int)
    y_pred.extend(preds.flatten())
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
ensemble_model = load_model('/content/drive/MyDrive/Models/Ensemble_tb_model.keras')

In [ ]:
resnet_model = load_model('/content/drive/MyDrive/Models/resnet_tb_model.keras')
vgg_model = load_model('/content/drive/MyDrive/Models/vgg_tb_model.keras')
mobile_model = load_model('/content/drive/MyDrive/Models/mobile_tb_model.keras')

def grad_cam_single_model(model, img_array, last_conv_layer_name, pred_index=None):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

# ====== Preprocess Image ======
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    return img_array

# ====== Predict & Explain ======
def predict_and_explain(img_path):
    img_array = preprocess_image(img_path)

    # 1️⃣ Prediction from ensemble
    pred_prob = ensemble_model.predict(img_array)[0][0]
    pred_class = "Tuberculosis" if pred_prob > 0.5 else "Normal"
    print(f"Ensemble Prediction: {pred_class} ({pred_prob:.4f})")

    # 2️⃣ Grad-CAM from one base model (ResNet50 here)
    last_conv_layer_name = "conv5_block3_out"  # ResNet50 last conv layer
    heatmap = grad_cam_single_model(resnet_model, img_array, last_conv_layer_name)

    # 3️⃣ Overlay Heatmap
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)

    # Show
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.title("Original")
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM (ResNet50)")
    plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


In [ ]:
predict_and_explain("/content/drive/MyDrive/sample_xray2.jpeg")